In [26]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm
import re
import sys
from pathlib import Path
import glob
import subprocess

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.39.11
Rasterio version: 1.4.3


In [27]:
# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing_local import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities_local import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


In [28]:
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'
local_path = "local-files"

###

EVENT_NAME = "202409_TropicalStorm_Francine"
product = "black_marble"

In [29]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

keys = [x.replace(f"drcs_activations/{EVENT_NAME}/{product}/", "") for x in get_all_s3_keys(s3_client, BUCKET, f"drcs_activations/{EVENT_NAME}/{product}", ".tif")] if s3_client else []

keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files


['BMHD/BatonRouge/VNP46A2_BRDFCorrected.A2024256.Sep122024.BatonRouge.tif',
 'BMHD/BatonRouge/VNP46A2_BRDFCorrected.A2024257.Sep132024.BatonRouge.tif',
 'BMHD/BatonRouge/VNP46A2_BRDFCorrected.A2024258.Sep142024.BatonRouge.tif',
 'BMHD/BatonRouge/VNP46A2_BRDFCorrected.A2024259.Sep152024.BatonRouge.tif',
 'BMHD/BatonRouge/VNP46A3.A2024153.June2024.BatonRouge.tif',
 'BMHD/Houma/VNP46A2_BRDFCorrected.A2024256.Sep122024.Houma.tif',
 'BMHD/Houma/VNP46A2_BRDFCorrected.A2024257.Sep132024.Houma.tif',
 'BMHD/Houma/VNP46A2_BRDFCorrected.A2024258.Sep142024.Houma.tif',
 'BMHD/Houma/VNP46A2_BRDFCorrected.A2024259.Sep152024.Houma.tif',
 'BMHD/Houma/VNP46A3.A2024153.June2024.Houma.tif',
 'BMHD/Lafayette/VNP46A2_BRDFCorrected.A2024256.Sep122024.Lafayette.tif',
 'BMHD/Lafayette/VNP46A2_BRDFCorrected.A2024257.Sep132024.Lafayette.tif',
 'BMHD/Lafayette/VNP46A2_BRDFCorrected.A2024258.Sep142024.Lafayette.tif',
 'BMHD/Lafayette/VNP46A2_BRDFCorrected.A2024259.Sep152024.Lafayette.tif',
 'BMHD/Lafayette/VNP46A3

In [30]:
# Download the desired files to a local directory with a known path
local_file_dir = os.path.abspath(f"./{local_path}")
if not os.path.exists(local_file_dir):
    os.mkdir(local_file_dir)
for key in keys:
    subprocess.run([
        "aws",
        "s3",
        "cp",
        f"s3://nasa-disasters/drcs_activations/{EVENT_NAME}/{product}/{key}",
        local_file_dir], check = True)

download: s3://nasa-disasters/drcs_activations/202409_TropicalStorm_Francine/black_marble/BMHD/BatonRouge/VNP46A2_BRDFCorrected.A2024256.Sep122024.BatonRouge.tif to local-files/VNP46A2_BRDFCorrected.A2024256.Sep122024.BatonRouge.tif
download: s3://nasa-disasters/drcs_activations/202409_TropicalStorm_Francine/black_marble/BMHD/BatonRouge/VNP46A2_BRDFCorrected.A2024257.Sep132024.BatonRouge.tif to local-files/VNP46A2_BRDFCorrected.A2024257.Sep132024.BatonRouge.tif
download: s3://nasa-disasters/drcs_activations/202409_TropicalStorm_Francine/black_marble/BMHD/BatonRouge/VNP46A2_BRDFCorrected.A2024258.Sep142024.BatonRouge.tif to local-files/VNP46A2_BRDFCorrected.A2024258.Sep142024.BatonRouge.tif
download: s3://nasa-disasters/drcs_activations/202409_TropicalStorm_Francine/black_marble/BMHD/BatonRouge/VNP46A2_BRDFCorrected.A2024259.Sep152024.BatonRouge.tif to local-files/VNP46A2_BRDFCorrected.A2024259.Sep152024.BatonRouge.tif
download: s3://nasa-disasters/drcs_activations/202409_TropicalStorm_

In [31]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()


# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

In [32]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [54]:
def make_regex_dict(keys, regexes, products):
    ret = {}
    for i in range(len(products)):
        matches = []
        for key in keys:
            filename = key.split("/")[-1]
            match = re.search(regexes[i], filename)
            if match is not None:
                matches.append(key)
        if matches != []:
            ret[products[i]] = matches
    return ret

In [76]:
def create_cog_filename(filename, event):
    if re.search(r".*DNB_BRDF-Corrected.*.tif", filename) is not None:
        sname = filename.split("/")[-1].replace(".tif", "").split(".")
        date = datetime.strptime(sname[1], "A%Y%j")
        new_dt_format = date.strftime("%Y-%m-%d_day")
        cog_filename = f"{event}_{sname[0]}_{sname[3]}_{sname[4]}_{new_dt_format}.tif"

    elif re.search(r".*[A-Z0-9]{7}_BRDFCorrected.*.tif", filename) is not None:
        sname = filename.split("/")[-1].replace(".tif", "").split(".")
        date = datetime.strptime(sname[1], "A%Y%j")
        new_dt_format = date.strftime("%Y-%m-%d_day")
        cog_filename = f"{event}_{sname[0]}_{sname[3]}_{new_dt_format}.tif"

    elif re.search(r".*June2024.*.tif", filename) is not None:
        if "Mosaic" in filename:
            sname = filename.split("/")[-1].replace(".tif", "").split(".")
            date = datetime.strptime(sname[1], "A%Y%j")
            new_dt_format = date.strftime("%Y-%m_monthly")
            cog_filename = f"{event}_{sname[0]}_{sname[1]}_{sname[3]}_{new_dt_format}.tif"
        else:
            sname = filename.split("/")[-1].replace(".tif", "").split(".")
            date = datetime.strptime(sname[1], "A%Y%j")
            new_dt_format = date.strftime("%Y-%m_monthly")
            cog_filename = f"{event}_{sname[0]}_{sname[3]}_{new_dt_format}.tif"

    elif re.search(r".*_Cloud_.*.tif", filename) is not None:
        sname = filename.split("/")[-1].replace(".tif", "").split(".")
        date = datetime.strptime(sname[1], "A%Y%j")
        new_dt_format = date.strftime("%Y-%m-%d_day")
        cog_filename = f"{event}_{sname[0]}_{sname[3]}_{sname[4]}_{new_dt_format}.tif"

    else:
        print(f"{filename} not caught by regexes!")
        return None
    
    return cog_filename

In [77]:
local_keys = [x for x in glob.glob(f"{local_path}/*") if x.endswith(".tif")]

reg_keys = make_regex_dict(local_keys, [r".*DNB_BRDF-Corrected.*.tif", r".*[A-Z0-9]{7}_BRDFCorrected.*.tif", r".*June2024.*.tif", r".*_Cloud_.*.tif"], ["dnb", "brdf-corrected", "hd", "qf-cloud"])

local_keys

['local-files/VNP46A3.A2024153.202419014383.June2024.Mosaic_C2_Clip.tif',
 'local-files/VNP46A2_BRDFCorrected.A2024256.Sep122024.BatonRouge.tif',
 'local-files/VNP46A2_BRDFCorrected.A2024257.Sep132024.BatonRouge.tif',
 'local-files/VNP46A2_BRDFCorrected.A2024258.Sep142024.BatonRouge.tif',
 'local-files/VNP46A2_BRDFCorrected.A2024259.Sep152024.BatonRouge.tif',
 'local-files/VNP46A3.A2024153.June2024.BatonRouge.tif',
 'local-files/VNP46A2_BRDFCorrected.A2024256.Sep122024.Houma.tif',
 'local-files/VNP46A2_BRDFCorrected.A2024257.Sep132024.Houma.tif',
 'local-files/VNP46A2_BRDFCorrected.A2024258.Sep142024.Houma.tif',
 'local-files/VNP46A2_BRDFCorrected.A2024259.Sep152024.Houma.tif',
 'local-files/VNP46A3.A2024153.June2024.Houma.tif',
 'local-files/VNP46A2_BRDFCorrected.A2024256.Sep122024.Lafayette.tif',
 'local-files/VNP46A2_BRDFCorrected.A2024257.Sep132024.Lafayette.tif',
 'local-files/VNP46A2_BRDFCorrected.A2024258.Sep142024.Lafayette.tif',
 'local-files/VNP46A2_BRDFCorrected.A2024259.Sep

In [78]:
print(reg_keys)
for k, v in reg_keys.items():
    for filename in v:
        print(create_cog_filename(filename, EVENT_NAME))

{'dnb': ['local-files/VNP46A2.A2024256.Sep12.DNB_BRDF-Corrected_NTLC2.Mosaic_C2_Clip.tif', 'local-files/VNP46A2.A2024259.Sep15.DNB_BRDF-Corrected_NTLC2.Mosaic_C2_Clip.tif', 'local-files/VNP46A2.A2024257.Sep13.DNB_BRDF-Corrected_NTLC2.Mosaic_C2_Clip.tif', 'local-files/VNP46A2.A2024258.Sep14.DNB_BRDF-Corrected_NTLC2.Mosaic_C2_Clip.tif'], 'brdf-corrected': ['local-files/VNP46A2_BRDFCorrected.A2024256.Sep122024.BatonRouge.tif', 'local-files/VNP46A2_BRDFCorrected.A2024257.Sep132024.BatonRouge.tif', 'local-files/VNP46A2_BRDFCorrected.A2024258.Sep142024.BatonRouge.tif', 'local-files/VNP46A2_BRDFCorrected.A2024259.Sep152024.BatonRouge.tif', 'local-files/VNP46A2_BRDFCorrected.A2024256.Sep122024.Houma.tif', 'local-files/VNP46A2_BRDFCorrected.A2024257.Sep132024.Houma.tif', 'local-files/VNP46A2_BRDFCorrected.A2024258.Sep142024.Houma.tif', 'local-files/VNP46A2_BRDFCorrected.A2024259.Sep152024.Houma.tif', 'local-files/VNP46A2_BRDFCorrected.A2024256.Sep122024.Lafayette.tif', 'local-files/VNP46A2_BRDF

In [79]:
def simple_process_files(file_list, rename_func, target_dir, event):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    print("Testing filenams:")
    for filename in file_list:
        print(f"  {rename_func(filename, event)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        #"raw_data_bucket": BUCKET,
        #"raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{event}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(local_download_path, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            local_download_path, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=file_list,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=event,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

In [80]:
if not os.path.exists(os.path.abspath("./output")):
    os.mkdir(os.path.abspath("./output"))
if not os.path.exists(os.path.abspath("./reproj")):
    os.mkdir(os.path.abspath("./reproj"))
for k, v in reg_keys.items():
    results = simple_process_files(file_list = v, rename_func = create_cog_filename, target_dir = f"Blackmarble/{k}", event = EVENT_NAME)

Testing filenams:
  202409_TropicalStorm_Francine_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_Clip_2024-09-12_day.tif
  202409_TropicalStorm_Francine_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_Clip_2024-09-15_day.tif
  202409_TropicalStorm_Francine_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_Clip_2024-09-13_day.tif
  202409_TropicalStorm_Francine_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_Clip_2024-09-14_day.tif
Configuration loaded:
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Blackmarble/dnb

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202409_TropicalStorm_Francine

[1/4] Processing: local-files/VNP46A2.A2024256.Sep12.DNB_BRDF-Corrected_NTLC2.Mosaic_C2_Clip.tif
   Output filename: 202409_TropicalStorm_Francine_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_Clip_2024-09-12_day.tif
   [CACHE HIT] Using local file: local-files/VNP46A2.A2024256.Sep12.DNB_BRDF-Corrected_NTLC2.Mosaic_C2_Clip.tif
   [MEMORY] Initial: 327.2 MB
   [REPROJECT] Already

Reading input: /tmp/tmpsdlz9bi5_temp.tif

Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp_a3yaoci.tif


   [CONVERT] Using rio-cogeo for conversion...
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Blackmarble/dnb/202409_TropicalStorm_Francine_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_Clip_2024-09-12_day.tif
   [MEMORY] Final: 369.0 MB (Change: +41.8 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_TropicalStorm_Francine_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_Clip_2024-09-12_day.tif

[2/4] Processing: local-files/VNP46A2.A2024257.Sep13.DNB_BRDF-Corrected_NTLC2.Mosaic_C2_Clip.tif
   Output filename: 202409_TropicalStorm_Francine_VNP46A2_DNB_BRDF-Corrected_NTLC2_Mosaic_C2_Clip_2024-09-13_day.tif
   [CACHE HIT] Using local file: local-files/VNP46A2.A2024257.Sep13.DNB_BRDF-Corrected_NTLC2.Mosaic_C2_Clip.tif
   [MEMORY] Initial: 365.2 MB
 

Reading input: /tmp/tmpqkec33zw_temp.tif



   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=-999.9000244140625, max=1885.8419189453125, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [PREDICTOR] Data type: float32, using PREDICTOR=3
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


...

✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_TropicalStorm_Francine_VNP46A2_QF_Cloud_C2_Mosaic_Clip_2024-09-12_day.tif

[2/4] Processing: local-files/VNP46A2.A2024257.Sep132024.QF_Cloud_C2.Mosaic_Clip.tif
   Output filename: 202409_TropicalStorm_Francine_VNP46A2_QF_Cloud_C2_Mosaic_Clip_2024-09-13_day.tif
   [CACHE HIT] Using local file: local-files/VNP46A2.A2024257.Sep132024.QF_Cloud_C2.Mosaic_Clip.tif
   [MEMORY] Initial: 684.6 MB
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=1, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT

Reading input: /tmp/tmp6fy23_iw_temp.tif

Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpyrfpze1w.tif


✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_TropicalStorm_Francine_VNP46A2_QF_Cloud_C2_Mosaic_Clip_2024-09-13_day.tif

[3/4] Processing: local-files/VNP46A2.A2024258.Sep142024.QF_Cloud_C2.Mosaic_Clip.tif
   Output filename: 202409_TropicalStorm_Francine_VNP46A2_QF_Cloud_C2_Mosaic_Clip_2024-09-14_day.tif
   [CACHE HIT] Using local file: local-files/VNP46A2.A2024258.Sep142024.QF_Cloud_C2.Mosaic_Clip.tif
   [MEMORY] Initial: 684.6 MB
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=1, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT

Reading input: /tmp/tmp98kt1xhl_temp.tif

Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp11zjhdse.tif


✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_TropicalStorm_Francine_VNP46A2_QF_Cloud_C2_Mosaic_Clip_2024-09-14_day.tif

[4/4] Processing: local-files/VNP46A2.A2024259.Sep152024.QF_Cloud_C2.Mosaic_Clip.tif
   Output filename: 202409_TropicalStorm_Francine_VNP46A2_QF_Cloud_C2_Mosaic_Clip_2024-09-15_day.tif
   [CACHE HIT] Using local file: local-files/VNP46A2.A2024259.Sep152024.QF_Cloud_C2.Mosaic_Clip.tif
   [MEMORY] Initial: 684.6 MB
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=1, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT

In [81]:
subprocess.run(["rm", "-r", f"{local_path}"], check = True)
subprocess.run(["rm", "-r", os.path.abspath("./output")], check = True)
subprocess.run(["rm", "-r", os.path.abspath("./reproj")], check = True)

CompletedProcess(args=['rm', '-r', '/home/jovyan/conversion_scripts/convert-files-and-move/2024/reproj'], returncode=0)